# 🗂️ Employee Salary Dataset — EDA
---
📌 **Goal:** Predict salary for Level 6.5 using multiple ML models \
📊 **Dataset:** 10 rows | 3 columns → Position, Level, Salary \
✅ **Status:** Clean data, no nulls


In [2]:
import pandas as pd

In [3]:
dataset = pd.read_csv("emp_sal.csv")
print("shape:", dataset.shape)
print("first 5 rows \n:", dataset.head())
print("null count:", dataset.isnull().sum())

shape: (10, 3)
first 5 rows 
:                Position  Level  Salary
0  Jr Software Engineer      1   45000
1  Sr Software Engineer      2   50000
2             Team Lead      3   60000
3               Manager      4   80000
4            Sr manager      5  110000
null count: Position    0
Level       0
Salary      0
dtype: int64


# 📈 Model 1 — Linear Regression (Baseline)
---
🔍 **What it does:** Draws a single straight line through all data points
⚠️ **Limitation:** Salary grows non-linearly — straight line struggles at extremes
🎯 **Hyperparameter tuning:** None (no major hyperparameters)

> 📉 Expected: Low R² because salary curve is not a straight line

In [4]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

# prepare X and y
X = dataset.iloc[:,1:2]  # Level column, 2D
y = dataset.iloc[:,2]  # Salary column

# train linear regression
lin_reg = LinearRegression()
lin_reg.fit(X, y)

# predict on same X (we have only 10 rows, no train/test split needed)
y_pred = lin_reg.predict(X)

# print R2 and MAE
print("R2:", round(r2_score(y, y_pred), 4))
print("MAE:", round(mean_absolute_error(y, y_pred), 2))

R2: 0.669
MAE: 128454.55


# 🌀 Model 2 — Polynomial Regression
---
🔍 **What it does:** Bends the line into a curve using degree parameter
💡 **Key idea:** Higher degree = more bends = better fit (but watch out for overfitting!)

🎛️ **Hyperparameter tuned:** `degree` → [2, 3, 4, 5]

> ⚠️ Higher degree = memorizes data, may fail on unseen inputs

In [9]:
from sklearn.preprocessing import PolynomialFeatures

results = []
for degree in range(1,6):
    poly = PolynomialFeatures(degree=degree)
    X_poly = poly.fit_transform(X)

    # step 2: fit linear regression on X_poly
    lin_reg_poly = LinearRegression()
    lin_reg_poly.fit(X_poly, y)

    # step 3: predict and score
    y_pred_poly = lin_reg_poly.predict(X_poly)
    r2 = round(r2_score(y, y_pred_poly), 4)
    mae = round(mean_absolute_error(y, y_pred_poly), 2)

    results.append({"degree": degree, "R2": r2, "MAE": mae})

pd.DataFrame(results)

,degree,R2,MAE
0,1,0.6690,128454.55
1,2,0.9162,70218.18
2,3,0.9812,34790.21
3,4,0.9974,12681.82
4,5,0.9998,3360.84


In [10]:
for degree in [2, 3, 4, 5]:
    poly = PolynomialFeatures(degree=degree)
    X_poly = poly.fit_transform(X)
    model = LinearRegression()
    model.fit(X_poly, y)
    pred = model.predict(poly.fit_transform([[6.5]]))
    print(f"Degree {degree} → predicted salary: {round(pred[0])}")

Degree 2 → predicted salary: 189498
Degree 3 → predicted salary: 133259
Degree 4 → predicted salary: 158862
Degree 5 → predicted salary: 174878


# 🏘️ Model 3 — K-Nearest Neighbours Regressor
---
🔍 **What it does:** Finds K closest data points and averages their salary
💡 **Real world analogy:** Ask your neighbours what they earn → take average

🎛️ **Hyperparameters tuned:**
| Param | Options | Meaning |
|-------|---------|---------|
| `n_neighbors` | 1,2,3,4,5 | How many neighbours to ask |
| `weights` | uniform, distance | Equal vote OR closer = more vote |
| `p` | 1, 2 | Manhattan vs Euclidean distance |

In [12]:
from sklearn.neighbors import KNeighborsRegressor

results_knn = []

for k in [1, 2, 3, 4, 5]:
    for w in ['uniform', 'distance']:
        for p in [1, 2]:
            knn = KNeighborsRegressor(
                n_neighbors=k,
                weights=w,
                p=p
            )
            knn.fit(X, y)
            y_pred_knn = knn.predict(X)

            r2 = round(r2_score(y,y_pred_knn), 4)
            mae = round(mean_absolute_error(y,y_pred_knn), 2)
            pred_6 = round(knn.predict([[6.5]])[0], 2)

            results_knn.append({
                "n_neighbors": k,
                "weights": w,
                "p": p,
                "R2": r2,
                "MAE": mae,
                "Pred@6.5": pred_6
            })

pd.DataFrame(results_knn)

D:\Data_Science\01_Practice\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but KNeighborsRegressor was fitted with feature names
  warnings.warn(
D:\Data_Science\01_Practice\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but KNeighborsRegressor was fitted with feature names
  warnings.warn(
D:\Data_Science\01_Practice\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but KNeighborsRegressor was fitted with feature names
  warnings.warn(
D:\Data_Science\01_Practice\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but KNeighborsRegressor was fitted with feature names
  warnings.warn(
D:\Data_Science\01_Practice\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but KNeighborsRegressor was fitted w

,n_neighbors,weights,p,R2,MAE,Pred@6.5
0,1,uniform,1,1.0000,0.00,150000.00
1,1,uniform,2,1.0000,0.00,150000.00
2,1,distance,1,1.0000,0.00,150000.00
3,1,distance,2,1.0000,0.00,150000.00
4,2,uniform,1,0.9053,48000.00,175000.00
5,2,uniform,2,0.9053,48000.00,175000.00
6,2,distance,1,1.0000,0.00,175000.00
7,2,distance,2,1.0000,0.00,175000.00
8,3,uniform,1,0.7874,57166.67,153333.33
9,3,uniform,2,0.7874,57166.67,153333.33


# 🌳 Model 4 — Decision Tree Regressor
---
🔍 **What it does:** Splits data by asking yes/no questions on Level value,
reaching a salary answer at each leaf (end point)

💡 **Analogy:** Game of 20 questions — keeps splitting until it finds the salary bucket

🎛️ **Hyperparameters tuned:**
| Param | Options | Meaning |
|-------|---------|---------|
| `max_depth` | 1,2,3,4,5 | How many splits deep |
| `min_samples_split` | 2,3,4 | Min rows needed to split further |
| `criterion` | squared_error, poisson | How split quality is measured |

⚠️ **Watch out:** Deep trees = overfitting (memorizes all 10 rows!)

In [13]:
from sklearn.tree import DecisionTreeRegressor

results_dt = []

for depth in [1, 2, 3, 4, 5]:
    for mss in [2, 3, 4]:
        for crit in ['squared_error', 'poisson']:
            dt = DecisionTreeRegressor(
                max_depth=depth,
                min_samples_split=mss,
                criterion=crit
            )
            dt.fit(X, y)
            y_pred_dt = dt.predict(X)

            r2 = round(r2_score(y,y_pred_dt), 4)
            mae = round(mean_absolute_error(y,y_pred_dt), 2)
            pred_6 = round(dt.predict([[6.5]])[0], 2)

            results_dt.append({
                "depth": depth,
                "mss": mss,
                "crit": crit,
                "R2": r2,
                "MAE": mae,
                "Pred@6.5": pred_6
            })

pd.DataFrame(results_dt)

D:\Data_Science\01_Practice\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but DecisionTreeRegressor was fitted with feature names
  warnings.warn(
D:\Data_Science\01_Practice\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but DecisionTreeRegressor was fitted with feature names
  warnings.warn(
D:\Data_Science\01_Practice\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but DecisionTreeRegressor was fitted with feature names
  warnings.warn(
D:\Data_Science\01_Practice\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but DecisionTreeRegressor was fitted with feature names
  warnings.warn(
D:\Data_Science\01_Practice\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but DecisionTreeRegressor wa

,depth,mss,crit,R2,MAE,Pred@6.5
0,1,2,squared_error,0.7764,105375.00,124375.00
1,1,2,poisson,0.7764,105375.00,124375.00
2,1,3,squared_error,0.7764,105375.00,124375.00
3,1,3,poisson,0.7764,105375.00,124375.00
4,1,4,squared_error,0.7764,105375.00,124375.00
5,1,4,poisson,0.7764,105375.00,124375.00
6,2,2,squared_error,0.9835,29000.00,82500.00
7,2,2,poisson,0.9820,27066.67,216666.67
8,2,3,squared_error,0.8286,79000.00,82500.00
9,2,3,poisson,0.8271,77066.67,216666.67


# 🌲🌲🌲 Model 5 — Random Forest Regressor
---
🔍 **What it does:** Builds N Decision Trees on random data subsets,
averages all predictions for final answer

💡 **Analogy:** Ask 100 doctors instead of 1 → average opinion wins!

🎛️ **Hyperparameters tuned:**
| Param | Options | Meaning |
|-------|---------|---------|
| `n_estimators` | 10,50,100,200 | How many trees in the forest |
| `max_depth` | 2,3,4,5 | How deep each tree can grow |
| `min_samples_split` | 2,3,4 | Min rows needed to split a node |

⚠️ **Watch out:** Too many deep trees = overfitting still possible!

In [18]:
from sklearn.ensemble import RandomForestRegressor

results_rf = []

for n in [10, 50, 100, 200]:
    for depth in [2, 3, 4, 5]:
        for mss in [2, 3, 4]:
            rf = RandomForestRegressor(
                n_estimators=n,
                max_depth=depth,
                min_samples_split= mss,
                random_state=42
            )
            rf.fit(X, y)
            y_pred_rf = rf.predict(X)

            r2 = round(r2_score(y,y_pred_rf), 4)
            mae = round(mean_absolute_error(y,y_pred_rf), 2)
            pred_6 = round(rf.predict([[6.5]])[0], 2)

            results_rf.append({
                "n_estimator": n,
                "max_depth": depth,
                "mss": mss,
                "R2": r2,
                "MAE": mae,
                "Pred@6.5": pred_6
            })

pd.DataFrame(results_rf)

D:\Data_Science\01_Practice\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
D:\Data_Science\01_Practice\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
D:\Data_Science\01_Practice\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
D:\Data_Science\01_Practice\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
D:\Data_Science\01_Practice\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor wa

,n_estimator,max_depth,mss,R2,MAE,Pred@6.5
0,10,2,2,0.9283,46473.33,179113.10
1,10,2,3,0.8443,54306.67,179113.10
2,10,2,4,0.8422,56814.29,172898.81
3,10,3,2,0.9354,36763.33,180000.00
4,10,3,3,0.8455,50385.00,197833.33
5,10,3,4,0.8433,55244.05,180327.38
6,10,4,2,0.9358,34750.00,180000.00
7,10,4,3,0.8456,50135.00,197833.33
8,10,4,4,0.8433,55244.05,180327.38
9,10,5,2,0.9358,34700.00,180000.00


# 📊 Excel Report — Model Comparison
---
📁 **Two sheets exported:**
- 🏆 `Leaderboard` → Final ranking of all models
- 📋 `Details` → All hyperparameter combinations tried

🛠️ **Library used:** `openpyxl`

In [21]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment

wb = Workbook()

# ── SHEET 1: LEADERBOARD ──────────────────────────
ws1 = wb.active
ws1.title = "Leaderboard"

headers = ["Rank", "Model", "Best R²", "Best MAE", "Pred@6.5"]
leaderboard_data = [
    [1, "Polynomial",    0.9998, 3360,    174878],
    [2, "Decision Tree", 0.9977, 9666,    175000],
    [3, "Random Forest", 0.9485, 31123,   172666],
    [4, "KNN",           0.9053, 48000,   175000],
    [5, "Linear Reg",    0.6690, 128454,  "N/A"],
]
medals = ["🥇", "🥈", "🥉", "4️⃣", "5️⃣"]

ws1.append(headers)
for cell in ws1[1]:
    cell.font = Font(bold=True, name="Arial", color="FFFFFF")
    cell.fill = PatternFill("solid", start_color="2F4F8F")
    cell.alignment = Alignment(horizontal="center")

for i, row in enumerate(leaderboard_data):
    row[0] = medals[i]
    ws1.append(row)
    for cell in ws1[i + 2]:
        cell.alignment = Alignment(horizontal="center")
        cell.font = Font(name="Arial")
        if i == 0:
            cell.fill = PatternFill("solid", start_color="FFD700")

for col, width in zip("ABCDE", [8, 18, 10, 12, 12]):
    ws1.column_dimensions[col].width = width


# ── SHEET 2: DETAILS ──────────────────────────────
ws2 = wb.create_sheet("Details")

det_headers = ["Model", "Hyperparameter", "Value", "R²", "MAE"]
ws2.append(det_headers)
for cell in ws2[1]:
    cell.font = Font(bold=True, name="Arial", color="FFFFFF")
    cell.fill = PatternFill("solid", start_color="2F4F8F")
    cell.alignment = Alignment(horizontal="center")

detail_data = []

# Polynomial ✅ variable name is 'results'
for row in results:
    detail_data.append(["Polynomial", "degree", row["degree"], row["R2"], row["MAE"]])

# KNN ✅ keys match
for row in results_knn:
    detail_data.append(["KNN", f"k={row['n_neighbors']},w={row['weights']},p={row['p']}", "-", row["R2"], row["MAE"]])

# Decision Tree ✅ fixed: depth, mss, crit
for row in results_dt:
    detail_data.append(["Decision Tree", f"depth={row['depth']},mss={row['mss']}", row["crit"], row["R2"], row["MAE"]])

# Random Forest ✅ fixed: n_estimator, mss
for row in results_rf:
    detail_data.append(["Random Forest", f"n={row['n_estimator']},depth={row['max_depth']}", f"mss={row['mss']}", row["R2"], row["MAE"]])

for row in detail_data:
    ws2.append(row)

for col, width in zip("ABCDE", [15, 30, 15, 10, 12]):
    ws2.column_dimensions[col].width = width

wb.save("ML_Model_Comparison.xlsx")
print("✅ Excel saved: ML_Model_Comparison.xlsx")

✅ Excel saved: ML_Model_Comparison.xlsx
